# UMAP Video Generator

Generate combined videos that show a dataset view alongside the saved UMAP watershed density map, with a red marker indicating the current classified position.

Use the configuration cell below to change dataset selection, paths, and prototype settings. Leave the implementation cells unchanged unless you are modifying the rendering logic.

## Configuration

Edit this section only if you want to change notebook inputs or output behavior.

What you can change here:
- `DATASET_SELECTION` to choose `hive1_upper`, `hive2_upper`, `hive2_lower`, or `all`.
- `PROTOTYPE_FRAME_LIMIT` to control how many source-video frames are processed during prototyping.
- `LAYOUT`, `FPS`, and output paths if you want a different video layout or destination.
- The `video_alias` entries if your alias-backed source videos move.


In [18]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd()
PROJECT_DIR = BASE_DIR
RESULTS_DIR = PROJECT_DIR / "Results" / "WideNormalisation" / "UMAP"
PROJECTIONS_DIR = PROJECT_DIR / "Results" / "WideNormalisation" / "Projections"
WSHED_PATH = RESULTS_DIR / "zVals_wShed_groups.mat"
OUTPUT_DIR = PROJECT_DIR / "outputVideos" / "umap_video"
LAYOUT = "side_by_side"  # or "top_bottom"
FPS = 20
VIDEO_PREFIX = "dataset_umap"
FRAME_LIMIT = None
BUCKET = "ObsHiveABC"
DOWNLOAD_RESOLUTION = 600
SOURCE_VIDEO_SCALE = 0.4
UMAP_PANEL_FIGSIZE = (8, 8)
UMAP_PANEL_DPI = 180
UMAP_PANEL_TITLE_FONTSIZE = 16
UMAP_PANEL_AXIS_LABELSIZE = 14
UMAP_PANEL_TICK_LABELSIZE = 12

# Dataset selection and alias-backed source videos
DATASET_CONFIGS = {
    "hive1_upper": {
        "hive_nb": 1,
        "ihl": "upper",
        "start_ts": pd.Timestamp('2024-09-04 16:00:00').tz_localize("UTC"),
        "projection_source_id": 1,
        "video_alias": "/Users/cyrilmonette/Desktop/EPFL 2018-2026/PhD - Mobots/RHC/RHCVisualisation/outputVideos/Ethograms/dataset2",
    },
    "hive2_upper": {
        "hive_nb": 2,
        "ihl": "upper",
        "start_ts": pd.Timestamp('2024-10-02 18:00:00').tz_localize("UTC"),
        "projection_source_id": 0,
        "video_alias": "/Users/cyrilmonette/Desktop/EPFL 2018-2026/PhD - Mobots/RHC/RHCVisualisation/outputVideos/Ethograms/dataset1",
    },
    "hive2_lower": {
        "hive_nb": 2,
        "ihl": "lower",
        "start_ts": pd.Timestamp('2024-10-02 18:00:00').tz_localize("UTC"),
        "projection_source_id": 2,
        "video_alias": "/Users/cyrilmonette/Desktop/EPFL 2018-2026/PhD - Mobots/RHC/RHCVisualisation/outputVideos/Ethograms/dataset1",
    },
}

DATASET_SELECTION = "hive2_lower"  # choose one key above or "all"
selected_datasets = [DATASET_SELECTION] if DATASET_SELECTION != "all" else list(DATASET_CONFIGS.keys())

## Imports

Import libraries for data handling, plotting, image processing, and video creation. The helper module provides the reusable UMAP/video frame functions.

In [19]:
import sys
import importlib
import numpy as np
import cv2
from tqdm.auto import tqdm
import imageio.v2 as imageio

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import umap_video_utils as umap_video_utils_module
umap_video_utils_module = importlib.reload(umap_video_utils_module)

compose_panels = umap_video_utils_module.compose_panels
load_umap_points_from_projection_file = umap_video_utils_module.load_umap_points_from_projection_file
load_watershed_artifacts = umap_video_utils_module.load_watershed_artifacts
render_umap_panel = umap_video_utils_module.render_umap_panel
generateVideoFromFrames = umap_video_utils_module.generateVideoFromFrames
resolve_macos_alias = umap_video_utils_module.resolve_macos_alias
load_projection_timeline = umap_video_utils_module.load_projection_timeline
load_umap_data = umap_video_utils_module.load_umap_data
blackout_unused_hive_half = umap_video_utils_module.blackout_unused_hive_half

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## Load UMAP Embedding, Density Map, and Classifier Outputs

Load the precomputed watershed density background and the border coordinates needed to place the current timestamp on the UMAP map.

In [20]:
wshed_artifacts = load_watershed_artifacts(str(WSHED_PATH))

## Create Frame Overlay for the Current Classification Point

Draw a red dot on the UMAP density map at the classified position for the current frame and prepare the visualization for frame-by-frame updates.

In [21]:
def render_umap_frame_for_timestamp(timestamp:pd.Timestamp, coord:np.ndarray, *, wshed_artifacts: dict, region=None) -> np.ndarray:
    return render_umap_panel(
        coord,
        wshed_artifacts["density"],
        extent=wshed_artifacts["extent"],
        wbounds=wshed_artifacts["wbounds"],
        barycenters=None,
        title="UMAP density map",
        timestamp=timestamp,
        region=region,
        figsize=UMAP_PANEL_FIGSIZE,
        dpi=UMAP_PANEL_DPI,
        title_fontsize=UMAP_PANEL_TITLE_FONTSIZE,
        axis_labelsize=UMAP_PANEL_AXIS_LABELSIZE,
        tick_labelsize=UMAP_PANEL_TICK_LABELSIZE,
    )


## Render Combined Video for a Single Dataset

Assemble the video frames with the UMAP panel, write the final video to disk, and expose this as a function that accepts a dataset identifier.

In [22]:
def render_dataset_video(dataset_name: str, *, layout: str = LAYOUT, fps: int = FPS) -> str:
    dataset_config = DATASET_CONFIGS[dataset_name]
    source_video_path = resolve_macos_alias(dataset_config["video_alias"])
    frame_interval = pd.Timedelta(minutes=20) # For all videos, frames are 20' apart
    umap_data = load_umap_data(DATASET_CONFIGS, PROJECTIONS_DIR, dataset_name)

    cap = cv2.VideoCapture(str(source_video_path))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    if FRAME_LIMIT is not None:
        frame_count = min(frame_count, FRAME_LIMIT)

    full_timeline = pd.date_range(start=dataset_config["start_ts"], periods=frame_count, freq=frame_interval)

    reader = imageio.get_reader(str(source_video_path))

    def frame_iterator(dts: pd.DatetimeIndex):
        progress = tqdm(total=frame_count, desc=f'"{dataset_name}" video frames', unit='frame')
        # compute video timestamps from known start and fixed cadence (20 minutes per frame)

        for idx, (_dt, video_frame) in enumerate(zip(dts, reader)):
            video_frame = np.asarray(video_frame)
            if video_frame.ndim == 2:
                video_frame = np.stack([video_frame] * 3, axis=-1)
            if video_frame.shape[2] == 4:
                video_frame = video_frame[:, :, :3]
            if SOURCE_VIDEO_SCALE != 1.0:
                new_width = max(1, int(video_frame.shape[1] * SOURCE_VIDEO_SCALE))
                new_height = max(1, int(video_frame.shape[0] * SOURCE_VIDEO_SCALE))
                video_frame = cv2.resize(video_frame, (new_width, new_height), interpolation=cv2.INTER_AREA)
            video_frame = blackout_unused_hive_half(video_frame, dataset_config["ihl"] )

            # lookup a matching UMAP point for the _dt. If it does not exist, use None.
            umap_coord = None
            try:
                row = umap_data[umap_data["real_timestamps"] == _dt].iloc[0]
                umap_coord = np.array([row["umap_x"], row["umap_y"]], dtype=float)
            except IndexError:
                umap_coord = None

            # render the UMAP panel with the video timestamp and optional point overlay
            umap_frame = render_umap_frame_for_timestamp(
                _dt,
                umap_coord,
                wshed_artifacts=wshed_artifacts,
            )
            yield compose_panels(video_frame, umap_frame, layout=layout)
            progress.update(1)
        progress.close()

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    generate_video_name = f"{VIDEO_PREFIX}_{dataset_name}"
    try:
        output_path = generateVideoFromFrames(frame_iterator(full_timeline), dest=str(OUTPUT_DIR), name=generate_video_name, fps=fps)
    finally:
        reader.close()
    return output_path


In [23]:
dataset_name = 'hive1_upper'
umap_df = load_umap_data(dataset_cfg=DATASET_CONFIGS, projections_dir=PROJECTIONS_DIR, dataset_name=dataset_name)
print(f"Filtered UMAP timeline shape: {umap_df.shape}, start: {umap_df['real_timestamps'].min()}, end: {umap_df['real_timestamps'].max()}")

# show a few entries
print('\nfiltered timeline head:')
print(umap_df.head())

Loading UMAP data for dataset 'hive1_upper' with projection_source_id=1 from /Users/cyrilmonette/Desktop/EPFL 2018-2026/PhD - Mobots/PdS/(25b) Ethograms/Ethogram-Generation/Results/WideNormalisation/Projections
Filtered UMAP timeline shape: (3587, 3), start: 2024-09-20 23:10:00+00:00, end: 2024-10-28 04:40:00+00:00

filtered timeline head:
      umap_x     umap_y           real_timestamps
0  13.618742  77.619522 2024-09-20 23:10:00+00:00
1  13.645033  77.589485 2024-09-20 23:20:00+00:00
2  13.700061  77.563728 2024-09-20 23:30:00+00:00
3  13.734835  77.553284 2024-09-20 23:40:00+00:00
4  13.629657  77.600082 2024-09-20 23:50:00+00:00


## Loop Over the Selected Dataset

Add batch-processing logic to generate videos for each dataset using the same rendering pipeline and configurable settings.

In [24]:
video_outputs = {}
for dataset_name in selected_datasets:
    video_outputs[dataset_name] = render_dataset_video(dataset_name, layout=LAYOUT, fps=FPS)

video_outputs


Loading UMAP data for dataset 'hive2_lower' with projection_source_id=2 from /Users/cyrilmonette/Desktop/EPFL 2018-2026/PhD - Mobots/PdS/(25b) Ethograms/Ethogram-Generation/Results/WideNormalisation/Projections


"hive2_lower" video frames:   0%|          | 0/1852 [00:00<?, ?frame/s]

{'hive2_lower': '/Users/cyrilmonette/Desktop/EPFL 2018-2026/PhD - Mobots/PdS/(25b) Ethograms/Ethogram-Generation/outputVideos/umap_video/dataset_umap_hive2_lower.mp4'}